In [2]:
import lightgbm as lgb
import pandas as pd
import json
data = pd.read_csv("CFBPollDataFinal.csv")
data = data.drop(columns=["total_wins_before", "total_losses_before", "team", "rank", "poll_point_diff"])

FEATURE_COLS = [col for col in data.columns if col not in ["next_week_points", "year", "week"]]

X = data[FEATURE_COLS].copy()
y = data["next_week_points"]

cats = ["conference", "opponent_conference", "result", "bias", "opponent"]

for col in cats:
  X[col] = X[col].astype("category")
    
model = lgb.LGBMRegressor(learning_rate=0.01, n_estimators=1500, max_depth=6, random_state=42)
model.fit(X, y, categorical_feature=cats)
model.booster_.save_model("CFBLightGBM.txt")

with open("feature_cols.json", "w") as f:
    json.dump(FEATURE_COLS, f)


[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001076 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1778
[LightGBM] [Info] Number of data points in the train set: 69572, number of used features: 16
[LightGBM] [Info] Start training from score 34.361367
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, bes

In [7]:
X_pred = data[FEATURE_COLS].copy()

for col in cats:
    X_pred[col] = X[col].cat.set_categories(X[col].cat.categories)

data["predicted_points"] = model.predict(X_pred)

data["model_rank"] = (
    data.groupby(["year", "week"])["predicted_points"]
      .rank(method="min", ascending=False)
      .astype(int)
)

data.head()

,year,week,conference,poll_points,result,pts_scored,pts_allowed,pt_diff,opponent,opponent_conference,...,opponent_losses,opponent_points,next_week_points,bias,undefeated,total_wins_after,total_losses_after,streak,predicted_points,model_rank
0,2014,1,Southland,0,L,37.0,38.0,-1.0,Georgia State,Sun Belt,...,0.0,0,0,0,0.0,0.0,1.0,-1.0,-0.418158,360
1,2014,1,Mountain West,0,W,44.0,16.0,28.0,Nicholls,Southland,...,0.0,0,0,0,1.0,1.0,0.0,1.0,0.019455,45
2,2014,1,Mid-American,0,W,41.0,0.0,41.0,Howard,MEAC,...,0.0,0,0,0,1.0,1.0,0.0,1.0,0.019455,45
3,2014,1,SEC,0,W,33.0,23.0,10.0,West Virginia,Big 12,...,0.0,0,0,5,1.0,1.0,0.0,1.0,0.183264,23
4,2014,1,SWAC,0,L,13.0,47.0,-34.0,North Carolina A&T,MEAC,...,0.0,0,0,0,0.0,0.0,1.0,-1.0,-0.008601,202


In [9]:
# data["predicted_points"] = model.predict(data[FEATURE_COLS])
# data["model_rank"] = (
#     data.groupby(["year", "week"])["predicted_points"]
#       .rank(method="min", ascending=False)
#       .astype(int)
# )
# data["actual_rank_num"] = pd.to_numeric(data["rank"], errors="coerce")

# df["rank_diff"] = (
#     df["model_rank"] - df["actual_rank_num"]
# ).abs()